BRONZE LAYER

In [0]:
df=spark.read.json("/Volumes/workspace/training/store_records/product_1000_records.jsonl")

df.write.format("delta").mode("overwrite").save("/Volumes/workspace/training/store_records/product_1000_records.delta")

df_delta=spark.read.format("delta").load("/Volumes/workspace/training/store_records/product_1000_records.delta")
df_delta.display()

SILVER LAYER

In [0]:
from pyspark.sql.functions import regexp_replace, expr, col, when

df_silver = (df_delta.withColumn("price", expr("regexp_replace(price, '[^0-9]', '')"))).withColumn("price",expr("try_cast(price as int)"))

df_silver = (df_silver.withColumn("qty",col("inventory.qty")).withColumn("warehouse",col("inventory.warehouse")).withColumn("supplier_id",col("supplier.id")).withColumn("supplier_name",col("supplier.name")).drop("inventory","supplier")
.withColumn("unit_price", col("price") / col("qty"))
.withColumn("unit_price", when(col("unit_price") < 100, col("unit_price") * 10).otherwise(col("unit_price")))
.withColumn(
    "category",
    when(col("unit_price") < 100, "D")
    .when((col("unit_price") >= 100) & (col("unit_price") <= 200), "C")
    .when((col("unit_price") > 200) & (col("unit_price") <= 500), "B")
    .when((col("unit_price") > 500) & (col("unit_price") <= 1000), "A")
    .when(col("unit_price") > 1000, "A+")
))

df_silver = df_silver.dropDuplicates()


df_silver.display()

df_silver.write.format("delta").mode("overwrite").saveAsTable("workspace.training.product_1000_records_silver")

In [0]:
# from pyspark.sql.window import Window
# from pyspark.sql.functions import row_number, col

# SCD1_TABLE_1 = "workspace.training.product_scd1_1"
# dedup_window = Window.partitionBy("product_num").orderBy(col("unit_price").desc())
# df_scd1_src = (df_silver
#     .withColumn("row_num", row_number().over(dedup_window))
#     .filter(col("row_num") == 1)
#     #.drop("row_num")
# )

# (df_scd1_src.write.format("delta")
#     .mode("ignore")
#     .saveAsTable(SCD1_TABLE_1))

# df_scd1_src.createOrReplaceTempView("product_scd1_src_view")

# spark.sql(f"""
#     MERGE INTO {SCD1_TABLE} AS tgt
#     USING product_scd1_src_view AS src
#     ON tgt.product_num = src.product_num
#     WHEN MATCHED THEN UPDATE SET *
#     WHEN NOT MATCHED THEN INSERT *
# """)

# spark.table(SCD1_TABLE_1).display()

In [0]:
from pyspark.sql.functions import current_timestamp, lit, expr

SCD2_TABLE = "workspace.training.product_scd2"

# Add SCD2 columns
df_scd2_src = df_silver.withColumn("start_date", expr("current_timestamp() - INTERVAL 1 DAY")) \
    .withColumn("end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

(df_scd2_src.write.format("delta")
    .mode("ignore")
    .saveAsTable(SCD2_TABLE))

df_scd2_src.createOrReplaceTempView("product_scd2_src_view")

spark.sql(f"""
MERGE INTO {SCD2_TABLE} AS tgt
USING product_scd2_src_view AS src
ON tgt.product_num = src.product_num AND tgt.is_current = TRUE
WHEN MATCHED AND (
    tgt.unit_price <> src.unit_price AND tgt.unit_price < src.unit_price
) THEN
  UPDATE SET
    tgt.end_date = current_timestamp(),
    tgt.is_current = FALSE
WHEN MATCHED AND (
    tgt.unit_price <> src.unit_price AND tgt.unit_price > src.unit_price
) THEN
  UPDATE SET
    tgt.start_date = current_timestamp() + INTERVAL 1 DAY
WHEN NOT MATCHED THEN
  INSERT *
""")

spark.table(SCD2_TABLE).display()

In [0]:
from pyspark.sql.functions import count, col

df = spark.table(SCD2_TABLE)
dupes = (
    df.groupBy("product_num")
    .agg(count("*").alias("count"), expr("count(distinct unit_price) as unit_price_count"))
    .filter(col("count") > 1)
    .filter(col("unit_price_count") > 1)
    .select("product_num")
)
df.join(dupes, "product_num").orderBy(col("product_num")).display()